# Introduction to Image Segmentation

- Image segmentation is the process of partitioning an image into multiple segments (sets of pixels).
- Segmentation is different from classification:
  - Classification: classify the whole image into one category
  - Segmentation: segment the image into multiple regions

- Segmentation is different from object detection:
  - Object detection: detect the presence of objects in an image and locate their bounding boxes
  - Segmentation: segment the image into multiple regions, including the boundaries of the objects

## Segmentation Types

:::: {.columns}
::: {.column width="70%"}
**Semantic Segmentation**

- Each pixel is assigned a single category or label.
- Does not distinguish between instances of the same class.

**Instance Segmentation**

- Assigns a unique identifier to each instance of an object.
- Background regions are not explicitly segmented.

**Panoptic Segmentation**

- Combines semantic and instance segmentation.
- Provides a class label for each pixel while also distinguishing between instances.

:::
::: {.column width="30%"}
![](./img/segmentation-types.png){width=80%}
:::
::::

# Transposed Convolutions
Transposed convolutions (also known as deconvolutions or fractionally-strided convolutions) are essential operations in image segmentation architectures like U-Net:

- **Purpose**: Upsampling feature maps to increase spatial dimensions
- **Operation**: Reverses the spatial transformation of a normal convolution
- **Key characteristics**:
  - Learns parameters for upsampling (unlike simple interpolation)
  - Increases spatial dimensions while potentially reducing channels
  - Used in decoder sections of segmentation networks


## Transposed Convolution Example {.scrollable}

- In this example, we have a $2 \times 2$ kernel applied to a $3 \times 3$ input tensor.
- A stride of $2$ results in an output tensor with dimensions $6 \times 6$.

![](./img/transposed-convolution.gif)


In [30]:
#| echo: false

import imageio.v2 as imageio
from pathlib import Path

# Define paths to images for the gif
img_dir = Path('./img/transposed-convolution')
img_paths = list(img_dir.iterdir())
img_paths.sort()
img_paths

# Read images and create gif
images = [imageio.imread(p) for p in img_paths]
imageio.mimsave('./img/transposed-convolution.gif', images, duration=2000)

## PyTorch Transposed Convolution

- [A good explanation of transposed convolutions](https://towardsdatascience.com/understand-transposed-convolutions-and-build-your-own-transposed-convolution-layer-from-scratch-4f5d97b2967/)

In [31]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Input tensor [batch_size, channels, height, width]
img = torch.Tensor([[[[1, 0, 2], [3, 4, 0], [0, 5, -1]]]])
print("Input shape:", img.shape)  # [1, 1, 3, 3]

# Kernel tensor [in_channels, out_channels, kernel_height, kernel_width]
kernel = torch.Tensor([[[[1, 2], [3, 4]]]])
print("Kernel shape:", kernel.shape) 

# Transposed convolution
output = F.conv_transpose2d(img, kernel, stride=2, padding=0)
print("Output shape:", output.shape)
print("Output Tensor:", output)

Input shape: torch.Size([1, 1, 3, 3])
Kernel shape: torch.Size([1, 1, 2, 2])
Output shape: torch.Size([1, 1, 6, 6])
Output Tensor: tensor([[[[ 1.,  2.,  0.,  0.,  2.,  4.],
          [ 3.,  4.,  0.,  0.,  6.,  8.],
          [ 3.,  6.,  4.,  8.,  0.,  0.],
          [ 9., 12., 12., 16.,  0.,  0.],
          [ 0.,  0.,  5., 10., -1., -2.],
          [ 0.,  0., 15., 20., -3., -4.]]]])


# U-Net

- [U-Net](https://arxiv.org/pdf/1505.04597) is a convolutional neural network (CNN) designed for biomedical image segmentation.
- Uses an encoder-decoder structure with skip connections and transposed convolutions.
- Efficient for training on small datasets.

**Key Components:**

1. **Encoder (Contracting Path):** Extracts features using convolution and pooling layers.
2. **Bottleneck:** Connects the encoder and decoder.
3. **Decoder (Expanding Path):** Uses transposed convolutions to reconstruct spatial information.
4. **Skip Connections:** Preserve fine details by linking encoder and decoder layers.

## U-Net Encoder & Decoder Architecture

### **Encoder (Contracting Path)**

- Each step consists of:
  - Two **convolutional layers** (ReLU activation, small kernel size).
  - One **max pooling layer** (reduces spatial dimensions).
- Increases the number of feature channels while reducing the spatial size.

### **Decoder (Expanding Path)**

- Each step consists of:
  - **Transpose convolution** (upsampling spatial resolution).
  - **Concatenation** with corresponding feature maps from the encoder (skip connections).
  - Two **convolutional layers** (ReLU activation).
- Reconstructs the segmented image while retaining high-resolution details.



## Skip Connections & Final Output

### **Skip Connections**
- Directly link encoder layers to corresponding decoder layers.
- Help retain fine details lost during downsampling-preserving spatial information.
- Improve segmentation accuracy, especially for small structures.

### **Final Output Layer**
- A **1×1 convolution** reduces the feature maps to the number of segmentation classes.
- Typically followed by a **softmax** or **sigmoid** activation for classification.
    - **Softmax**: multi-class classification
    - **Sigmoid**: binary classification



## U-Net Diagram

[![](./img/unet-diagram.png)](https://arxiv.org/pdf/1505.04597#page=2)

## U-Net Loss Functions: BCE

- Best for tasks where object and background pixels are balanced.
- Struggles with highly imbalanced datasets (e.g., small objects in large images).
- Can only be used for binary segmentation.
- BCE is defined as:

$$
\mathcal{L}_{\text{BCE}} = -\frac{1}{N} \sum_{i} \left( y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right)
$$

- $y_i$ is the true label (0 or 1) for pixel $i$.
- $\hat{y}_i$ is the predicted probability for pixel $i$.
- $N$ is the number of pixels

## U-Net Loss Functions: CCE

- Categorical Cross Entropy (CCE) can be used for multi-class segmentation.
- CCE is defined as:

$$
\mathcal{L}_{\text{CCE}} = -\frac{1}{N} \sum_{i} \sum_{c} y_{i,c} \log(\hat{y}_{i,c})
$$

- $y_{i,c}$ is the one-hot encoded true label for pixel $i$.
- $\hat{y}_{i,c}$ is the predicted probability for class $c$ at pixel $i$.
- $N$ is the number of pixels


## U-Net Loss Functions: Weighted BCE

- The original paper uses weighted BCE loss to account for the imbalanced nature of the dataset.
- Medical images are often imbalanced (tumor or organ).

$$
\mathcal{L}_{\text{U-Net}} = -\sum_{i} w(x_i) \sum_{c} y_{i,c} \log(\hat{y}_{i,c})
$$

- $w(x_i)$ is a weight assigned to each pixel.
- $y_{i,c}$ is the one-hot encoded true label for pixel $i$.
- $\hat{y}_{i,c}$ is the predicted probability for class $c$ at pixel $i$.

## U-Net Loss Functions: Dice Loss

- Dice loss is based on the Dice Similarity Coefficient (DSC), which measures overlap between predicted and ground truth masks.
- Best for imbalanced datasets.
- Conceptually similar to Intersection over Union (IoU).

$$
\mathcal{L}_{Dice} = 1 - \frac{2 \sum_{i=1}^{N} y_i \hat{y}_i + \epsilon}{\sum_{i=1}^{N} y_i + \sum_{i=1}^{N} \hat{y}_i + \epsilon}
$$

- $y_i$ is the ground truth mask
- $\hat{y}_i$ is the predicted mask
- $N$ is the number of pixels
- $\epsilon$ is a small constant to avoid division by zero

## U-Net Loss Functions: Combined BCE + Dice Loss

- Combined BCE + Dice Loss is a weighted combination of BCE and Dice Loss.
- Best for general binary segmentation tasks.

$$
\mathcal{L} = \alpha \mathcal{L}_{BCE} + \beta \mathcal{L}_{Dice}
$$

- $\alpha$ is the weight for BCE
- $\beta$ is the weight for Dice
- $\alpha + \beta = 1$


## U-Net Loss Functions: Generalized Dice Loss *Supplemental*

- Generalized Dice Loss (GDice) is an extension of Dice Loss that handles multiple classes by weighting each class separately.
- Best for multi-class segmentation with class imbalance.

$$
\mathcal{L}_{GDice} = 1 - \frac{2 \sum_{c} w_c \sum_{i} y_{i,c} \hat{y}_{i,c}}{\sum_{c} w_c \sum_{i} (y_{i,c} + \hat{y}_{i,c})}
$$

- $w_c$ is a weight for class $c$
- $y_{i,c}$ is the one-hot encoded true label for pixel $i$
- $\hat{y}_{i,c}$ is the predicted probability for class $c$ at pixel $i$

## U-Net Loss Functions: Focal Loss *Supplemental*

- Focal Loss is designed to down-weight well-classified pixels and focus on hard-to-classify pixels.
- Best for datasets with a high imbalance between foreground and background.

$$
\mathcal{L}_{Focal} = -\frac{1}{N} \sum_{i} \sum_{c} \alpha (1 - \hat{y}_{i,c})^\gamma y_{i,c} \log(\hat{y}_{i,c})
$$

- $y_{i,c}$ is the one-hot encoded true label for pixel $i$
- $\hat{y}_{i,c}$ is the predicted probability for class $c$ at pixel $i$
- $\alpha$ is a class weighting factor
- $\gamma$ is a focus factor (typically 2)
- $N$ is the number of pixels



## Advantages and Applications of U-Net

### Applications of U-Net
- **Medical Image Segmentation** (e.g., tumor detection, organ segmentation).
- **Satellite Image Analysis** (e.g., land cover mapping).
- **Industrial Inspection** (e.g., defect detection). 
- **Self-driving Cars** (e.g., road and lane segmentation).

### **Advantages of U-Net**
- Works well with small datasets.
- Efficient training with data augmentation.
- Skip connections help preserve spatial details.
- Can be adapted for various segmentation tasks.

# ROI Align Overview

::::{ .columns}
:::{ .column width="60%"}

- ROI Align is a technique to map regions of interest to a uniform size.
- Replaces the ROI Pooling layer.
- Values are sampled from the feature map using bilinear interpolation.
- Used in segmentation models which might be sensitive to misalignments inherent to ROI pooling.
- Mask R-CNN uses ROI Align uses ROI Align over ROI Pooling.

:::
:::{ .column width="40%"}
[![](./img/mask-rcnn-roi-align.png)](https://arxiv.org/pdf/1703.06870v3#page=3)
:::
::::

In [20]:
#| echo: false

import imageio.v2 as imageio
import numpy as np
from pathlib import Path

# Define paths to images for the gif
img_dir = Path('./img/roi-align')
img_paths = list(img_dir.iterdir())
img_paths.sort()
img_paths

# Read images and create gif
images = [imageio.imread(p) for p in img_paths]
x, y = 0, 0
for img in images:
    # Find the largest dimension of the images
    x, y = max(x, img.shape[1]), max(y, img.shape[0])

padded_images = []
for img in images:
    img = np.pad(img, ((0, y-img.shape[0]), (0, x-img.shape[1]), (0, 0)), mode='constant')
    padded_images.append(img)

imageio.mimsave('./img/roi-align.gif', padded_images, duration=2000)

## ROI Align - Split ROI into Grid Cells

- User specifies the ROI and the output size.

![](./img/roi-align/roi-align-1.png)

## ROI Align - Define Sampling Points

- Find location of sampling points by splitting each grid cell into equidistant points.

![](./img/roi-align/roi-align-2.png)

## ROI Align - Nearest Neighbors

- Determine the sampling coordinates and 4 nearest pixels for each sampling point.

![](./img/roi-align/roi-align-3.png)

## ROI Align - Perform Bilinear Interpolation

- Perform bilinear interpolation to get the value of each sampling point, take mean to get final value.

![](./img/roi-align/roi-align-4.png)



## ROI Align - Final Output

- Combine the values of each grid cell to get the final output.

![](./img/roi-align/roi-align-5.png)



## ROI Align Full Workflow

![](./img/roi-align/roi-align-full.png)

## ROI Align PyTorch Example

In [1]:
import numpy as np
import torch
from torchvision.ops import roi_align

np_features = np.array(
    [
        [1,2,3,4,5],
        [6,7,8,9,10],
        [11,12,13,14,15],
        [16,17,18,19,20],
        [21,22,23,24,25]
    ]
)

output_size = (2, 2)
sampling_ratio = 2

# Define a tensor to perform ROI align on
torch_features = torch.tensor(np_features.reshape(1, 1, 5, 5), dtype=torch.float32)


# Define an RoI (batch_idx, x1, y1, x2, y2)
np_roi = np.array([-0.25, -0.25, 3, 3])
torch_roi = torch.tensor([[0, np_roi[0], np_roi[1], np_roi[2], np_roi[3]]], dtype=torch.float32)


# Perform ROI Align
output = roi_align(
    input=torch_features, 
    boxes=torch_roi, 
    output_size=output_size,
    spatial_scale=1.0,
    sampling_ratio=sampling_ratio,
)

print('The output of the ROI Align is:')
print(output)


The output of the ROI Align is:
tensor([[[[ 4.3750,  6.0000],
          [12.5000, 14.1250]]]])


## ROI Numpy Implementation {.scrollable}

In [33]:
# |echo: false

# Define a function to split a ROI into a grid of cells with sampling coordinates

# Define function to split a ROI into a grid of cells
def split_roi(roi, output_size=(2, 2), sampling_ratio=2):
    """
    Split a ROI into a grid of cells for given output size and sampling ratio

    roi: list of [x1, y1, x2, y2]
    output_size: tuple of (height, width)

    returns (tuple of numpy.ndarray) : coordinates of the grid cells (x1, y1, x2, y2)
    """

    # Get the height and width of the ROI
    x, y = np.meshgrid(np.linspace(roi[0], roi[2], (output_size[1]*(sampling_ratio+1))+1), np.linspace(roi[1], roi[3], (output_size[0]*(sampling_ratio+1))+1))
    
    return x, y

sampling_ratio = 2
x, y = split_roi(np_roi, output_size=(2,2), sampling_ratio=sampling_ratio)
# print(x)
# print(y)


In [2]:
# Define function to split a ROI into a grid of cells
def split_roi(roi, output_size=(2, 2)):
    """
    Split a ROI into a grid of cells for

    roi: list of [x1, y1, x2, y2]
    output_size: tuple of (height, width)

    returns (tuple of numpy.ndarray) : coordinates of the grid cells (x1, y1, x2, y2)
    """

    # Get the height and width of the ROI
    x, y = np.meshgrid(np.linspace(roi[0], roi[2], output_size[1]+1), np.linspace(roi[1], roi[3], output_size[0]+1))
    
    return x, y
    
x, y = split_roi(np_roi, output_size=(2,2))
print('The coordinates of the grid cells are:')
print('x:\n', x)
print('y:\n', y)

The coordinates of the grid cells are:
x:
 [[-0.25   1.375  3.   ]
 [-0.25   1.375  3.   ]
 [-0.25   1.375  3.   ]]
y:
 [[-0.25  -0.25  -0.25 ]
 [ 1.375  1.375  1.375]
 [ 3.     3.     3.   ]]


In [3]:
# Define x1, x2, y1, y2 for each cell in the grid
x1 = x[:-1, :-1]
x2 = x[1:, 1:]
y1 = y[:-1, :-1]
y2 = y[1:, 1:]

print('The x1, x2, y1, y2 for each cell in the grid are:')
print('x1:\n', x1)
print('x2:\n', x2)
print('y1:\n', y1)
print('y2:\n', y2)

The x1, x2, y1, y2 for each cell in the grid are:
x1:
 [[-0.25   1.375]
 [-0.25   1.375]]
x2:
 [[1.375 3.   ]
 [1.375 3.   ]]
y1:
 [[-0.25  -0.25 ]
 [ 1.375  1.375]]
y2:
 [[1.375 1.375]
 [3.    3.   ]]


In [4]:
# Define a function to return sampling points given a grid cell coordinates
def get_sampling_points(x1, x2, y1, y2, sampling_ratio=2):
    """
    Get sampling points from a grid cell

    x1 (float) : x-coordinate of the top-side of the grid cell
    x2 (float) : x-coordinate of the bottom-side of the grid cell
    y1 (float) : y-coordinate of the left-side of the grid cell
    y2 (float) : y-coordinate of the right-side of the grid cell
    sampling_ratio (int) : sampling points along a single axis of the grid cell.
    """
    x, y = np.meshgrid(np.linspace(x1, x2, sampling_ratio+2), np.linspace(y1, y2, sampling_ratio+2))
    
    return x[1:-1, 1:-1], y[1:-1, 1:-1] 

# Get sampling point coordinates for grid cell (0,0)
sx, sy = get_sampling_points(x1[0,0], x2[0,0], y1[0,0], y2[0,0], sampling_ratio=2)
print('The sampling point coordinates for the top left grid cell are:')
print('sx:\n', sx)
print('sy:\n', sy)

The sampling point coordinates for the top left grid cell are:
sx:
 [[0.29166667 0.83333333]
 [0.29166667 0.83333333]]
sy:
 [[0.29166667 0.29166667]
 [0.83333333 0.83333333]]


In [5]:
# Define a function to perform bilinear interpolation given sampling points and feature map values
def bilinear_interpolation(sx, sy, feature_map):
    """
    Perform bilinear interpolation given sampling points and feature map values
    for a single grid cell.

    sx (numpy.ndarray) : x-coordinates of the sampling points
    sy (numpy.ndarray) : y-coordinates of the sampling points
    feature_map (numpy.ndarray) : feature map
    """

    # Find the nearest integer coordinates of the sampling points  
    x1, x2 = np.floor(sx).astype(int), np.ceil(sx).astype(int)
    y1, y2 = np.floor(sy).astype(int), np.ceil(sy).astype(int)

    # print('x1:\n', x1)
    # print('x2:\n', x2)
    # print('y1:\n', y1)
    # print('y2:\n', y2)

    assert (x1<sx).all()
    assert (sx<x2).all()
    assert (y1<sy).all()
    assert (sy<y2).all()

    # Get the values of the feature map at the sampling points
    Q11 = feature_map[y1, x1]
    Q12 = feature_map[y2, x1]
    Q21 = feature_map[y1, x2]
    Q22 = feature_map[y2, x2]

    # print('Q11:\n', Q11)
    # print('Q12:\n', Q12)
    # print('Q21:\n', Q21)
    # print('Q22:\n', Q22)

    # Interpolate along the x-axis
    f_x1 = ((x2 - sx) / (x2 - x1)) * Q11 + ((sx - x1) / (x2 - x1)) * Q21
    f_x2 = ((x2 - sx) / (x2 - x1)) * Q12 + ((sx - x1) / (x2 - x1)) * Q22

    # print('f_x1:\n', f_x1)
    # print('f_x2:\n', f_x2)

    # Interpolate along the y-axis
    f_xy = (y2 - sy) / (y2 - y1) * f_x1 + (sy - y1) / (y2 - y1) * f_x2

    return f_xy


# Perform bilinear interpolation for a single grid cell
sx, sy = get_sampling_points(x1[0,0], x2[0,0], y1[0,0], y2[0,0], sampling_ratio=2)
fxy = bilinear_interpolation(sx, sy, np_features)
print('The values of the sampling points for the top left grid cell after bilinear interpolation are:')
print('fxy:\n', fxy)

print('The mean of the sampling points for the top left grid cell after bilinear interpolation is:')
print('fxy.mean():\n', fxy.mean())


The values of the sampling points for the top left grid cell after bilinear interpolation are:
fxy:
 [[2.75       3.29166667]
 [5.45833333 6.        ]]
The mean of the sampling points for the top left grid cell after bilinear interpolation is:
fxy.mean():
 4.375


In [7]:
# Write a function to perform ROI Align given a feature map and a ROI
def roi_align(feature_map, roi, output_size=(2, 2), sampling_ratio=2):
    """
    Perform ROI Align given a feature map and a ROI

    feature_map (numpy.ndarray) : feature map
    roi (list) : ROI
    output_size (tuple) : output size
    sampling_ratio (int) : sampling ratio
    """

    output_array = np.zeros(output_size)
    sampling_points = []
  
    # Split the ROI into a grid of cells
    x, y = split_roi(roi, output_size)

    # Define x1, x2, y1, y2 for each cell in the grid
    x1, x2 = x[:-1, :-1], x[1:, 1:]
    y1, y2 = y[:-1, :-1], y[1:, 1:]

    for i in range(output_size[0]): 
        for j in range(output_size[1]):
            # Get the sampling points
            sx, sy = get_sampling_points(x1[i,j], x2[i,j], y1[i,j], y2[i,j], sampling_ratio)

            # Perform bilinear interpolation
            fxy = bilinear_interpolation(sx, sy, feature_map)
            sampling_points.append(fxy)
            output_array[i,j] = fxy.mean()

    return output_array, sampling_points

# Test the roi_align function
output_array, sampling_points = roi_align(np_features, np_roi, output_size=(2, 2), sampling_ratio=2)
print('The ROI Align output array is:')
print(output_array)


The ROI Align output array is:
[[ 4.375  6.   ]
 [12.5   14.125]]


In [1]:
#| echo: false

# Check numpy implementation of ROI Align by hand
s1 = (((1-0.292)*(1-0.292))/((1-0)/(1-0)))*1 + (((0.292-0)*(1-0.292))/((1-0)/(1-0)))*2 + (((1-0.292)*(0.292-0))/((1-0)/(1-0)))*6 + (((0.292-0)*(0.292-0))/((1-0)/(1-0)))*7
s2 = (((1-0.833)*(1-0.292))/((1-0)/(1-0)))*1 + (((0.833-0)*(1-0.292))/((1-0)/(1-0)))*2 + (((1-0.833)*(0.292-0))/((1-0)/(1-0)))*6 + (((0.833-0)*(0.292-0))/((1-0)/(1-0)))*7
s3 = (((1-0.292)*(1-0.833))/((1-0)/(1-0)))*1 + (((0.292-0)*(1-0.833))/((1-0)/(1-0)))*2 + (((1-0.292)*(0.833-0))/((1-0)/(1-0)))*6 + (((0.292-0)*(0.833-0))/((1-0)/(1-0)))*7
s4 = (((1-0.833)*(1-0.833))/((1-0)/(1-0)))*1 + (((0.833-0)*(1-0.833))/((1-0)/(1-0)))*2 + (((1-0.833)*(0.833-0))/((1-0)/(1-0)))*6 + (((0.833-0)*(0.833-0))/((1-0)/(1-0)))*7

# print(s1)
# print(s2)
# print(s3)
# print(s4)

# Mask R-CNN

- [Mask R-CNN](https://arxiv.org/pdf/1703.06870) is a convolutional neural network (CNN) for object detection and **instance segmentation**.
- It extends Faster R-CNN by adding a mask prediction branch.
- It predicts **bounding boxes, class labels, and object masks**.
- Utilizes **Feature Pyramid Networks (FPN)** and **Region Proposal Networks (RPN)**.

![](./img/mask-r-cnn-overview.png)

## Mask R-CNN Architecture Overview

Mask R-CNN consists of:

- **Backbone (ResNet + FPN)**: Extracts multi-scale features.
- **Region Proposal Network (RPN)**: Proposes object candidate regions.
- **RoIAlign**: Extracts fixed-size feature maps from proposals.
- **Classification & Regression Head**: Assigns class labels and refines bounding boxes.
- **Segmentation Head**: Predicts pixel-wise masks for each object.

## Mask R-CNN Step-by-Step Workflow

### **Step 1: Feature Extraction**
- Input image is processed through a **CNN backbone (e.g., ResNet-50, ResNet-101)**.
- **Feature Pyramid Network (FPN)** generates multi-scale feature maps (**P2–P6**).

### **Step 2: Region Proposal Network (RPN)**
- **Anchors are generated** at different scales/aspect ratios.
- A $(3 \times 3)$ convolutional layer followed by two $(1 \times 1)$ convolutional layers predict objectness scores & bounding box deltas.
- **Top-k region proposals** are selected using non-maximum suppression (NMS) and passed to the next stage.

### **Step 3: RoIAlign (A Key Difference from Faster R-CNN)**
- Proposals are assigned to the **best FPN feature level**.
- **RoIAlign** extracts fixed-size (e.g., $(7 \times 7)$ and $(14 \times 14)$ feature maps) **without quantization artifacts**.



## Mask R-CNN Step-by-Step Workflow (continued)

### **Step 4: Bounding Box Classification & Refinement**
- The **classification head** predicts object classes.
- The **regression head** refines bounding box coordinates.
- **Non-Maximum Suppression (NMS)** is applied to remove duplicate detections.

### **Step 5: Mask Prediction**
- The **segmentation head** processes only the final selected boxes after NMS.
- Uses a **fully convolutional network (FCN)** to generate a **binary mask per object**.
- **Masks are generated independently for each class** - a mask is generated for each class.
    - *During inference only the mask for the predicted class is used.*

### **Step 6: Final Outputs**
- Refined **bounding boxes, class labels, and segmentation masks** are output.
- Masks are resized to fit the final detected object.

<!-- ## **4. Role of Feature Pyramid Networks (FPN)**
- **Why use FPN?** Improves detection of objects at different scales.
- Feature maps **P2–P6** provide hierarchical features for small & large objects.
- **Proposals are mapped to the appropriate FPN level** for RoIAlign.

---

## **5. Region Proposal Network (RPN) & RoIAlign**
- **RPN generates candidate bounding boxes** for potential objects.
- RoIAlign ensures that **feature extraction remains precise**.

---

## **6. Bounding Box Classification & Regression**
- The **classification head** assigns category labels.
- The **regression head** adjusts box coordinates.
- **NMS filters out redundant bounding boxes** before mask segmentation.

---

## **7. Segmentation Head & Mask Prediction**
- The **mask branch is separate from classification & regression**.
- It operates on the **final bounding boxes** after NMS.
- Generates a **binary mask (e.g., 28x28 resolution) for each object**.

---

## **8. Post-Processing (NMS, Mask Refinement)**
- NMS ensures **only the best bounding boxes are kept**.
- Predicted masks are resized to fit **refined bounding boxes**.

---

## **Summary**
✅ **Mask R-CNN extends Faster R-CNN with a segmentation head**.
✅ **FPN enhances multi-scale detection**.
✅ **RoIAlign improves feature extraction accuracy**.
✅ **Segmentation is performed only after final bounding box selection**.

Would you like additional slides on implementation details or training strategies? -->



## Mask R-CNN Diagram

![](./img/mask-r-cnn.png)

# Study Guide

## Image Segmentation Fundamentals

**Topics to review:**
- Semantic vs instance segmentation and what each predicts
- Typical input/output shapes for segmentation models
- Common losses (cross-entropy, Dice/IoU) and class imbalance issues
- Pixel accuracy vs IoU and when pixel accuracy can be misleading
- Why data augmentation is important for segmentation

**Example questions:**
- What is the difference between semantic and instance segmentation?
- Why can pixel accuracy be high even when segmentation quality is poor?
- When might Dice loss be preferred over cross-entropy?

## Transposed Convolutions

**Topics to review:**
- Why transposed convolutions are used for upsampling
- How stride and kernel size affect output size
- Checkerboard artifacts and how to reduce them (resize + conv)

**Example questions:**
- What does a transposed convolution do in a segmentation model?
- Why can transposed convolutions introduce checkerboard artifacts?

## U-Net

**Topics to review:**
- Encoder-decoder structure and skip connections
- How skip connections preserve spatial detail
- Where downsampling and upsampling occur in the network

**Example questions:**
- What problem do U-Net skip connections solve?
- How does U-Net combine global context with local detail?

## ROI Align

**Topics to review:**
- ROI Align vs ROI Pooling
- Why bilinear interpolation improves alignment
- Where ROI Align appears in detection/segmentation pipelines

**Example questions:**
- What issue does ROI Align fix compared to ROI Pooling?
- Why is precise alignment important for mask prediction?

## Mask R-CNN

**Topics to review:**
- How Mask R-CNN extends Faster R-CNN
- Separate branches for classification, bounding boxes, and masks
- The role of ROI Align in mask prediction

**Example questions:**
- What new head does Mask R-CNN add compared to Faster R-CNN?
- Why does Mask R-CNN need both bounding boxes and masks?
